In [2]:
%pip install supervision

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.idi.ntnu.no
Note: you may need to restart the kernel to use updated packages.


### Define paths and describe model

In [47]:
import os
import cv2
import supervision as sv
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm  # For progress bar
import shutil
import tkinter as tk
from tkinter import simpledialog
import datetime


# -------------  Change these and run all cells -------------------

dir_name = "example" 
yolov5_path = "/home/omtalmo/Olaf_TTK4265/TDT4265-Snow-pole-detection/yolov5_our_files/yolov5"
train_dir = "/home/omtalmo/Olaf_TTK4265/TDT4265-Snow-pole-detection/yolov5_our_files/yolov5/runs/train/exp5"
model_path = "/home/omtalmo/Olaf_TTK4265/TDT4265-Snow-pole-detection/yolov5_our_files/yolov5/runs/train/exp5/weights/best.pt"
dataset_path = "/home/omtalmo/Olaf_TTK4265/Poles/rgb"
output_dir = f"/home/omtalmo/Olaf_TTK4265/TDT4265-Snow-pole-detection/yolov5_our_files/results/{dir_name}"

# -------------------------------------------------------------------


# Local paths, list creation and folder creation
os.makedirs(output_dir, exist_ok=True)
valid_label_dir = f"{dataset_path}/labels/valid"

test_image_dir = f"{dataset_path}/images/test"
test_images = []
for img in os.listdir(test_image_dir):
    if img.upper().endswith('.PNG') or img.lower().endswith(('.png', '.jpg', '.jpeg')):
        test_images.append(os.path.join(test_image_dir, img))

valid_image_dir = f"{dataset_path}/images/valid"
valid_images = []
for img in os.listdir(valid_image_dir):
    if img.upper().endswith('.PNG') or img.lower().endswith(('.png', '.jpg', '.jpeg')):
        valid_images.append(os.path.join(valid_image_dir, img))

# Copy the contents of the training directory to the output directory
train_dir = "/home/omtalmo/Olaf_TTK4265/TDT4265-Snow-pole-detection/yolov12_our_files/yolov12/yolov12/runs/detect/train"
train_output_dir = os.path.join(output_dir, "train_results")
os.makedirs(train_output_dir, exist_ok=True)

# Add YOLOv5 to path
if not yolov5_path in sys.path:
    sys.path.append(yolov5_path)

# Import YOLOv5 modules
from models.experimental import attempt_load
from utils.general import non_max_suppression


# Load YOLOv5 model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = attempt_load(model_path, device=device)
model.eval()

# Copy the contents of the training directory
print(f"Copying training results from {train_dir} to {train_output_dir}...")
for item in os.listdir(train_dir):
    s = os.path.join(train_dir, item)
    d = os.path.join(train_output_dir, item)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)
print("Training results copied successfully!")

# Create a pop-up window to get the run description
def get_run_description():
    root = tk.Tk()
    root.title("Run Description")
    root.geometry("600x400")  # Set window size
    
    # Get current timestamp
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Add a label with instructions
    tk.Label(root, text="Beskriv modellen. Eller bare spam tastaturet, ditt valg:", 
             font=("Arial", 12), pady=10, padx=10).pack(fill='x')
    
    # Create a scrollable text area for multi-line input
    text_frame = tk.Frame(root)
    text_frame.pack(fill='both', expand=True, padx=20, pady=10)
    
    # Add scrollbar
    scrollbar = tk.Scrollbar(text_frame)
    scrollbar.pack(side='right', fill='y')
    
    # Create text widget with scrollbar
    text_box = tk.Text(text_frame, height=15, width=70, wrap='word', 
                       yscrollcommand=scrollbar.set, font=("Arial", 11))
    text_box.pack(side='left', fill='both', expand=True)
    scrollbar.config(command=text_box.yview)
    
    # Focus the text box
    text_box.focus_set()
    
    # Variable to store result
    description_var = tk.StringVar()
    
    # Functions for buttons
    def save_description():
        description_var.set(text_box.get("1.0", "end-1c"))
        root.destroy()
        
    def cancel():
        description_var.set("")
        root.destroy()
    
    # Button frame
    button_frame = tk.Frame(root)
    button_frame.pack(pady=15)
    
    # Add buttons
    tk.Button(button_frame, text="Save Description", command=save_description, 
              font=("Arial", 11), width=15, bg="#4CAF50", fg="white").pack(side='left', padx=10)
    tk.Button(button_frame, text="Cancel", command=cancel, 
              font=("Arial", 11), width=10).pack(side='left', padx=10)
    
    # Wait for user input
    root.wait_window()
    
    # Get the description
    description = description_var.get()
    
    if not description:  # User clicked Cancel or entered nothing
        description = "No description provided"
    
    # Save description to a text file
    desc_file = os.path.join(output_dir, "run_description.txt")
    with open(desc_file, 'w') as f:
        f.write(f"Run Description - {timestamp}\n")
        f.write(f"Dir Name: {dir_name}\n")
        f.write(f"Model: {str(model.ckpt_path)}\n\n")
        f.write(f"Description:\n{description}\n")
    
    print(f"Description saved to {desc_file}")
    return description

# Get and save the run description
run_description = get_run_description()

# Add YOLOv5 to path
yolov5_path = "/home/omtalmo/Olaf_TTK4265/TDT4265-Snow-pole-detection/yolov5_our_files/yolov5"
if not yolov5_path in sys.path:
    sys.path.append(yolov5_path)

# Import YOLOv5 modules
from models.experimental import attempt_load
from utils.general import non_max_suppression


# Load YOLOv5 model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = attempt_load(model_path, device=device)
model.eval()

Fusing layers... 
Model summary: 212 layers, 20852934 parameters, 0 gradients, 47.9 GFLOPs


Copying training results from /home/omtalmo/Olaf_TTK4265/TDT4265-Snow-pole-detection/yolov12_our_files/yolov12/yolov12/runs/detect/train to /home/omtalmo/Olaf_TTK4265/TDT4265-Snow-pole-detection/yolov5_our_files/results/example/train_results...
Training results copied successfully!


AttributeError: 'DetectionModel' object has no attribute 'ckpt_path'

### Run inference on all images and save in output_dir

In [37]:
import numpy as np

# Create annotators for visualization
bounding_box_annotator = sv.BoundingBoxAnnotator()
# We'll skip using label_annotator and draw text manually like in visualize_feature_maps_and_boxes

# Process all test images
print(f"Processing {len(test_images)} test images...")
for image_path in tqdm(test_images):
    # Load and process the image
    image = cv2.imread(image_path)
    if image is None:
        print(f"Warning: Could not load {image_path}")
        continue
    
    # Get YOLOv5 model input size
    imgsz = 640  # Default YOLOv5 input size
    
    # Prepare image for YOLOv5 inference
    # Resize and pad the image to maintain aspect ratio
    height, width = image.shape[:2]
    
    # Calculate scaling factors
    scale = min(imgsz / width, imgsz / height)
    new_width = int(width * scale)
    new_height = int(height * scale)
    
    # Resize the image
    resized_image = cv2.resize(image, (new_width, new_height))
    
    # Create a blank canvas with the target size
    input_image = np.zeros((imgsz, imgsz, 3), dtype=np.uint8)
    
    # Place the resized image on the canvas
    input_image[:new_height, :new_width, :] = resized_image
    
    # Keep track of the scale for later
    scale_factors = (scale, scale)
    pad_size = (0, 0, imgsz - new_width, imgsz - new_height)  # left, top, right, bottom

    # Convert to tensor and normalize
    img = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    img = torch.from_numpy(img).to(device)
    img = img.float() / 255.0  # Normalize 0-1
    if len(img.shape) == 3:
        img = img.permute(2, 0, 1)  # HWC to CHW (batch, channels, height, width)
    img = img.unsqueeze(0)  # Add batch dimension
    
    # Make predictions with YOLOv5
    with torch.no_grad():
        pred = model(img)  # Forward pass
        pred = non_max_suppression(pred, conf_thres=0.25, iou_thres=0.45)
    
    # Post-process predictions
    boxes, scores, class_ids = [], [], []
    if len(pred[0]) > 0:
        # Convert predictions back to original image coordinates
        for det in pred[0]:  # Process predictions for first (and only) image
            x1, y1, x2, y2, conf, cls_id = det
            
            # Convert from padded coordinates to original image coordinates
            if new_width < imgsz or new_height < imgsz:
                # Account for padding
                x1 = min(x1, new_width)
                x2 = min(x2, new_width)
                y1 = min(y1, new_height)
                y2 = min(y2, new_height)
                
            # Convert to original image coordinates
            x1, x2 = x1 / scale, x2 / scale
            y1, y2 = y1 / scale, y2 / scale
            
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            boxes.append([x1, y1, x2, y2])
            scores.append(float(conf))
            class_ids.append(int(cls_id))
    
    # Create sv.Detections object for supervision
    if boxes:
        detections = sv.Detections(
            xyxy=np.array(boxes),
            confidence=np.array(scores),
            class_id=np.array(class_ids),
        )
    else:
        # Empty detections
        detections = sv.Detections.empty()
    
    # Apply bounding box annotations only
    annotated_image = bounding_box_annotator.annotate(scene=image, detections=detections)
    
    # Manually add confidence text with two decimal places
    for i, (box, conf) in enumerate(zip(boxes, scores)):
        x1, y1, x2, y2 = box
        # Draw confidence text with 2 decimal places using OpenCV
        label = f"{conf:.2f}"
        cv2.putText(annotated_image, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    # Save the image result
    image_filename = Path(image_path).name
    output_path = os.path.join(output_dir, f"detection_{image_filename}")
    cv2.imwrite(output_path, annotated_image)

print(f"All results saved to {output_dir}")

# Optionally display a montage of some results (first 4 images)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, image_path in enumerate(test_images[:4]):
    if i >= 4:  # Limit to first 4 images
        break
        
    # Load the saved result
    image_filename = Path(image_path).name
    result_path = os.path.join(output_dir, f"detection_{image_filename}")
    
    if os.path.exists(result_path):
        result_img = cv2.imread(result_path)
        result_img_rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
        axes[i].imshow(result_img_rgb)
        axes[i].set_title(f"{image_filename}")
        axes[i].axis('off')
    
plt.tight_layout()
plt.show()

# Print summary 
print(f"Processed {len(test_images)} images")
print(f"Results saved to: {output_dir}")

Processing 46 test images...


SupervisionWarnings: BoundingBoxAnnotator is deprecated: `BoundingBoxAnnotator` is deprecated and has been renamed to `BoxAnnotator`. `BoundingBoxAnnotator` will be removed in supervision-0.26.0.


  0%|          | 0/46 [00:00<?, ?it/s]

All results saved to /home/omtalmo/Olaf_TTK4265/TDT4265-Snow-pole-detection/yolov5_our_files/results
Processed 46 images
Results saved to: /home/omtalmo/Olaf_TTK4265/TDT4265-Snow-pole-detection/yolov5_our_files/results


In [43]:
import os
import cv2
import numpy as np
import torch
from pathlib import Path
from tqdm.notebook import tqdm
import sys
import yaml
import datetime

# Import YOLOv5 modules
from models.experimental import attempt_load
from utils.general import non_max_suppression

# Load the model - reusing the one already loaded in the notebook
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = attempt_load(model_path, device=device)
model.eval()

# IoU calculation function
def calculate_iou(box1, box2):
    """
    Calculate IoU between two boxes
    box format: [x1, y1, x2, y2]
    """
    # Find intersecting box
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    # Check if there is an intersection
    if x2 < x1 or y2 < y1:
        return 0.0
    
    # Intersection area
    intersection_area = (x2 - x1) * (y2 - y1)
    
    # Union area
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - intersection_area
    
    # IoU
    return intersection_area / union_area if union_area > 0 else 0.0

# Function to load ground truth annotations
def load_annotations(image_path):
    """
    Load annotations for an image. Expected format is YOLO format (normalized xcenter, ycenter, width, height)
    """
    # Convert image path to label path
    image_path = Path(image_path)
    image_dir = image_path.parent
    image_dir_type = image_dir.name  # 'train', 'valid', or 'test'
    
    label_dir = Path(dataset_path) / "labels" / image_dir_type
    label_file = label_dir / f"{image_path.stem}.txt"
    
    boxes = []
    classes = []
    
    # Check if label file exists
    if label_file.exists():
        with open(label_file, 'r') as f:
            for line in f:
                data = line.strip().split()
                if len(data) >= 5:  # Should be class x y w h
                    class_id = int(data[0])
                    # Normalized coordinates (YOLO format)
                    x_center, y_center = float(data[1]), float(data[2])
                    width, height = float(data[3]), float(data[4])
                    
                    classes.append(class_id)
                    boxes.append([x_center, y_center, width, height])  # Store in normalized YOLO format
    
    return np.array(boxes), np.array(classes)

# Function to evaluate model on validation set - stats only
def evaluate_model_stats(model, valid_dir, iou_threshold=0.5, conf_threshold=0.5):
    """
    Evaluate model performance on validation set
    
    Args:
        model: YOLOv5 model
        valid_dir: Directory containing validation images
        iou_threshold: IoU threshold for a correct detection
        conf_threshold: Confidence threshold for model predictions
    
    Returns:
        Dictionary with evaluation metrics
    """
    # Find validation images
    valid_images = []
    for img in os.listdir(valid_dir):
        if img.upper().endswith('.PNG') or img.lower().endswith(('.png', '.jpg', '.jpeg')):
            valid_images.append(os.path.join(valid_dir, img))
    
    # Initialize counters
    total_gt_boxes = 0
    total_predictions = 0
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    
    # Track IoU values for correct predictions
    iou_values = []
    
    print(f"Evaluating model on {len(valid_images)} validation images...")
    for image_path in tqdm(valid_images):
        # Load image
        image = cv2.imread(image_path)
        if image is None:
            print(f"Could not load {image_path}, skipping")
            continue
        
        h, w = image.shape[:2]
        
        # Load ground truth annotations
        gt_boxes_norm, gt_classes = load_annotations(image_path)
        
        # Convert normalized gt_boxes to absolute pixel coordinates
        gt_boxes = []
        for box in gt_boxes_norm:
            x_center, y_center, box_width, box_height = box
            x1 = int((x_center - box_width/2) * w)
            y1 = int((y_center - box_height/2) * h)
            x2 = int((x_center + box_width/2) * w)
            y2 = int((y_center + box_height/2) * h)
            gt_boxes.append([x1, y1, x2, y2])
        
        # Update total ground truth boxes
        total_gt_boxes += len(gt_boxes)
        
        # Process image for YOLOv5
        imgsz = 640
        scale = min(imgsz / w, imgsz / h)
        new_width, new_height = int(w * scale), int(h * scale)
        
        # Resize image
        resized_image = cv2.resize(image, (new_width, new_height))
        
        # Create padded image
        input_image = np.zeros((imgsz, imgsz, 3), dtype=np.uint8)
        input_image[:new_height, :new_width, :] = resized_image
        
        # Convert to tensor and normalize
        img = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
        img = torch.from_numpy(img).to(device)
        img = img.float() / 255.0
        if len(img.shape) == 3:
            img = img.permute(2, 0, 1)  # HWC to CHW
        img = img.unsqueeze(0)  # Add batch dimension
        
        # Run model inference
        with torch.no_grad():
            pred = model(img)
            pred = non_max_suppression(pred, conf_thres=conf_threshold, iou_thres=0.5)
        
        # Process predictions
        pred_boxes = []
        pred_scores = []
        pred_classes = []
        
        if len(pred[0]) > 0:
            for det in pred[0]:
                x1, y1, x2, y2, conf, cls_id = det
                
                # Convert from padded coordinates to original image coordinates
                if new_width < imgsz or new_height < imgsz:
                    x1 = min(x1, new_width)
                    x2 = min(x2, new_width) 
                    y1 = min(y1, new_height)
                    y2 = min(y2, new_height)
                
                # Convert to original image coordinates
                x1, x2 = x1 / scale, x2 / scale
                y1, y2 = y1 / scale, y2 / scale
                
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
                pred_boxes.append([x1, y1, x2, y2])
                pred_scores.append(float(conf))
                pred_classes.append(int(cls_id))
        
        # Update total predictions
        total_predictions += len(pred_boxes)
        
        # Track which ground truth boxes are matched
        gt_matched = [False] * len(gt_boxes)
        
        # For each prediction, find the best matching ground truth
        for i, pred_box in enumerate(pred_boxes):
            best_iou = 0
            best_gt_idx = -1
            
            # Compare with all ground truth boxes
            for j, gt_box in enumerate(gt_boxes):
                if gt_matched[j]:  # Skip already matched ground truth
                    continue
                
                # Calculate IoU
                iou = calculate_iou(pred_box, gt_box)
                
                # Update best match
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = j
            
            # Check if the match is good enough
            if best_iou >= iou_threshold and best_gt_idx != -1:
                true_positives += 1
                gt_matched[best_gt_idx] = True
                iou_values.append(best_iou)
            else:
                false_positives += 1
        
        # Count unmatched ground truths as false negatives
        false_negatives += gt_matched.count(False)
    
    # Calculate metrics
    precision = true_positives / total_predictions if total_predictions > 0 else 0
    recall = true_positives / total_gt_boxes if total_gt_boxes > 0 else 0
    f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # Results
    results = {
        "total_gt_boxes": total_gt_boxes,
        "total_predictions": total_predictions,
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "mean_iou": np.mean(iou_values) if iou_values else 0
    }
    
    return results

# Run evaluation on validation set with stats only
valid_dir = f"{dataset_path}/images/valid"
results = evaluate_model_stats(model, valid_dir, iou_threshold=0.5, conf_threshold=0.5)

# Print results - stats only
print("\n===== Evaluation Results =====")
print(f"Total ground truth boxes: {results['total_gt_boxes']}")
print(f"Total predictions: {results['total_predictions']}")
print(f"True positives: {results['true_positives']}")
print(f"False positives: {results['false_positives']}")
print(f"False negatives: {results['false_negatives']}")
print(f"Precision: {results['precision']:.4f}")
print(f"Recall: {results['recall']:.4f}")
print(f"F1 Score: {results['f1_score']:.4f}")
print(f"Mean IoU of true positives: {results['mean_iou']:.4f}")

# Calculate additional metrics
print("\n===== Additional Metrics =====")
if results['total_gt_boxes'] > 0:
    miss_rate = results['false_negatives'] / results['total_gt_boxes']
    print(f"Miss rate: {miss_rate:.4f}")
else:
    miss_rate = float('nan')
    print("Miss rate: N/A")

if (results['false_positives'] + results['true_positives']) > 0:
    fdr = results['false_positives'] / (results['false_positives'] + results['true_positives'])
    print(f"False discovery rate: {fdr:.4f}")
else:
    fdr = float('nan')
    print("False discovery rate: N/A")

if (results['total_predictions'] + results['false_negatives']) > 0:
    total_accuracy = results['true_positives'] / (results['total_predictions'] + results['false_negatives'])
    print(f"Total accuracy: {total_accuracy:.4f}")
else:
    total_accuracy = float('nan')
    print("Total accuracy: N/A")

# Save results to text file
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
eval_results_path = os.path.join(output_dir, f"model_evaluation_results_{timestamp}.txt")

with open(eval_results_path, 'w') as f:
    f.write("===== YOLOv5 Model Evaluation Results =====\n")
    f.write(f"Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Model: {model_path}\n\n")
    
    f.write("===== Basic Metrics =====\n")
    f.write(f"Total ground truth boxes: {results['total_gt_boxes']}\n")
    f.write(f"Total predictions: {results['total_predictions']}\n")
    f.write(f"True positives: {results['true_positives']}\n")
    f.write(f"False positives: {results['false_positives']}\n")
    f.write(f"False negatives: {results['false_negatives']}\n")
    f.write(f"Precision: {results['precision']:.4f}\n")
    f.write(f"Recall: {results['recall']:.4f}\n")
    f.write(f"F1 Score: {results['f1_score']:.4f}\n")
    f.write(f"Mean IoU of true positives: {results['mean_iou']:.4f}\n\n")
    
    f.write("===== Additional Metrics =====\n")
    if results['total_gt_boxes'] > 0:
        f.write(f"Miss rate: {miss_rate:.4f}\n")
    else:
        f.write("Miss rate: N/A\n")
        
    if (results['false_positives'] + results['true_positives']) > 0:
        f.write(f"False discovery rate: {fdr:.4f}\n")
    else:
        f.write("False discovery rate: N/A\n")
        
    if (results['total_predictions'] + results['false_negatives']) > 0:
        f.write(f"Total accuracy: {total_accuracy:.4f}\n")
    else:
        f.write("Total accuracy: N/A\n")

print(f"\nEvaluation results saved to: {eval_results_path}")

Fusing layers... 
Model summary: 212 layers, 20852934 parameters, 0 gradients, 47.9 GFLOPs


Evaluating model on 92 validation images...


  0%|          | 0/92 [00:00<?, ?it/s]


===== Evaluation Results =====
Total ground truth boxes: 113
Total predictions: 70
True positives: 59
False positives: 11
False negatives: 54
Precision: 0.8429
Recall: 0.5221
F1 Score: 0.6448
Mean IoU of true positives: 0.7169

===== Additional Metrics =====
Miss rate: 0.4779
False discovery rate: 0.1571
Total accuracy: 0.4758

Evaluation results saved to: /home/omtalmo/Olaf_TTK4265/TDT4265-Snow-pole-detection/yolov5_our_files/results/model_evaluation_results_20250414_170701.txt


### Visualize all rejected bounding boxes

In [41]:
import os
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

def visualize_feature_maps_and_boxes(image_path, model, conf_threshold=0.00001):
    """
    Visualize YOLOv5's detection process at a lower level:
    1. Original image
    2. All potential boxes based on anchor points
    3. Low confidence predictions
    4. Final predictions

    Args:
        image_path: Path to the image
        model: YOLOv5 model
        conf_threshold: Extremely low threshold to see all possible candidates
    """
    # Load image
    image = cv2.imread(image_path)
    original_image = image.copy()
    h, w = image.shape[:2]
    
    # Use global device variable
    global device
    
    # Create figure with 2x2 grid
    fig, axes = plt.subplots(2, 2, figsize=(20, 16))
    fig.suptitle(f"YOLOv5 Deep Visualization", fontsize=16)
    
    # 1. Original image (top-left)
    axes[0, 0].imshow(cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB))
    axes[0, 0].set_title("Original Image")
    axes[0, 0].axis('off')
    
    # 2. Anchor points with primitive boxes (top-right)
    # Get the stride and anchor information from model
    try:
        # Alternative approach to get YOLOv5 anchors
        anchor_points = []
        anchor_img = original_image.copy()
        axes[0, 1].imshow(cv2.cvtColor(anchor_img, cv2.COLOR_BGR2RGB))
        
        # For YOLOv5, we need to access the model's modules differently
        # First, find the Detect module that contains anchors
        detect_layer = None
        for module in model.modules():
            if module.__class__.__name__ == 'Detect':
                detect_layer = module
                break
        
        if detect_layer is None:
            raise ValueError("Could not find YOLOv5 Detect layer")
            
        # Get strides from model (typically [8, 16, 32] for small, medium, large)
        strides = detect_layer.stride.tolist()  # Convert tensor to list
        if isinstance(strides, int):  # Handle the case where stride is a scalar
            strides = [strides]
        
        # Get anchors - in YOLOv5, these are stored in the detect layer
        anchors = detect_layer.anchors.cpu().numpy()  
        
        # Draw grid for each stride with different colors
        colors = ['red', 'green', 'blue', 'yellow', 'cyan']
        legend_elements = []
        
        # Process each stride (typically 3 scales)
        for i, stride in enumerate(strides):
            stride = int(stride)
            color = colors[i % len(colors)]
            
            # Create grid points for this stride
            x_points = np.arange(stride//2, w, stride)
            y_points = np.arange(stride//2, h, stride)
            
            # Track all anchor points
            for x in x_points:
                for y in y_points:
                    anchor_points.append((x, y))
                    
            # Sample some points to avoid overcrowding
            np.random.seed(42)  # For reproducibility
            x_samples = np.random.choice(x_points, min(10, len(x_points)), replace=False)
            y_samples = np.random.choice(y_points, min(10, len(y_points)), replace=False)
            
            # Draw anchor points as small circles
            for x in x_points:
                for y in y_points:
                    axes[0, 1].add_patch(plt.Circle((x, y), radius=2, color=color, alpha=0.3))
                    
            # Draw sample bounding boxes at selected anchor points
            for x in x_samples:
                for y in y_samples:
                    # Each anchor index corresponds to this stride's anchors
                    curr_anchors = anchors[i]
                    for anchor in curr_anchors:
                        # Convert normalized anchors to pixels
                        width = anchor[0] * stride  # anchor width * stride
                        height = anchor[1] * stride  # anchor height * stride
                        
                        rect = plt.Rectangle(
                            (x - width/2, y - height/2),
                            width, height,
                            linewidth=0.5,
                            edgecolor=color,
                            facecolor='none',
                            alpha=0.4
                        )
                        axes[0, 1].add_patch(rect)
            
            legend_elements.append(plt.Line2D([0], [0], marker='o', color=color, 
                                          label=f'Stride {stride}', markersize=8, linestyle='None'))
        
        axes[0, 1].legend(handles=legend_elements, loc='upper right')
        axes[0, 1].set_title(f"Anchor Points Grid ({len(anchor_points)} points) with Potential Boxes")
        axes[0, 1].axis('off')
        
    except Exception as e:
        print(f"Couldn't draw anchors due to: {e}")
        import traceback
        traceback.print_exc()  # Print full stack trace for debugging
        axes[0, 1].text(0.5, 0.5, f"Anchor visualization not available: {str(e)}",
                      horizontalalignment='center', verticalalignment='center', 
                      transform=axes[0, 1].transAxes)
        axes[0, 1].axis('off')
    
    # 3. Run prediction with very low confidence (bottom-left)
    # Prepare image for YOLOv5 inference
    imgsz = 640  # Default YOLOv5 input size
    
    # Calculate scaling factors
    scale = min(imgsz / w, imgsz / h)
    new_width = int(w * scale)
    new_height = int(h * scale)
    
    # Resize the image
    resized_image = cv2.resize(image, (new_width, new_height))
    
    # Create a blank canvas with the target size
    input_image = np.zeros((imgsz, imgsz, 3), dtype=np.uint8)
    
    # Place the resized image on the canvas
    input_image[:new_height, :new_width, :] = resized_image
    
    # Convert to tensor and normalize
    img = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    img = torch.from_numpy(img).to(device)  # Use global device instead of model.device
    img = img.float() / 255.0  # Normalize 0-1
    if len(img.shape) == 3:
        img = img.permute(2, 0, 1)  # HWC to CHW (batch, channels, height, width)
    img = img.unsqueeze(0)  # Add batch dimension
    
    # Run inference with extremely low confidence
    with torch.no_grad():
        pred = model(img)  # Forward pass
        pred_low_conf = non_max_suppression(pred, conf_thres=conf_threshold, iou_thres=0.45)
    
    # Get all boxes
    boxes_low_conf = []
    if len(pred_low_conf[0]) > 0:
        for det in pred_low_conf[0]:
            x1, y1, x2, y2, conf, cls_id = det
            
            # Convert from padded coordinates to original image coordinates
            if new_width < imgsz or new_height < imgsz:
                # Account for padding
                x1 = min(x1, new_width)
                x2 = min(x2, new_width)
                y1 = min(y1, new_height)
                y2 = min(y2, new_height)
            
            # Convert to original image coordinates
            x1, x2 = x1 / scale, x2 / scale
            y1, y2 = y1 / scale, y2 / scale
            
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            boxes_low_conf.append([x1, y1, x2, y2, float(conf), int(cls_id)])
    
    # Draw all detected boxes from explicit prediction
    low_conf_img = original_image.copy()
    
    # Count boxes by confidence ranges
    conf_ranges = {
        'very_low': [conf_threshold, 0.01],
        'low': [0.01, 0.1],
        'medium': [0.1, 0.5], 
        'high': [0.5, 1.0]
    }
    
    box_counts = {range_name: 0 for range_name in conf_ranges}
    
    for box in boxes_low_conf:
        x1, y1, x2, y2, conf, cls_id = box
        
        # Determine confidence range
        for range_name, (min_conf, max_conf) in conf_ranges.items():
            if min_conf <= conf < max_conf:
                box_counts[range_name] += 1
                break
                
        # Color based on confidence (green to red)
        color_b = max(0, int(255 * (1-conf)))  
        color_g = max(0, int(255 * conf))
        color_r = max(0, int(255 * (0.5 - abs(conf - 0.5)) * 2))  # Peak at 0.5 conf
        
        cv2.rectangle(low_conf_img, (x1, y1), (x2, y2), (color_b, color_g, color_r), 1)
    
    axes[1, 0].imshow(cv2.cvtColor(low_conf_img, cv2.COLOR_BGR2RGB))
    
    # Create legend text for confidence ranges
    conf_text = "\n".join([
        f"{range_name.replace('_', ' ').title()}: {count} boxes ({min_conf:.4f}-{max_conf:.1f})"
        for range_name, count in box_counts.items()
        for min_conf, max_conf in [conf_ranges[range_name]]
    ])
    
    total_boxes = sum(box_counts.values())
    axes[1, 0].text(10, 30, f"Total: {total_boxes} boxes\n{conf_text}", 
                    bbox=dict(facecolor='white', alpha=0.7), fontsize=9)
    
    axes[1, 0].set_title(f"All Raw Predictions: {len(boxes_low_conf)} boxes")
    axes[1, 0].axis('off')
    
    # 4. Final filtered predictions (bottom-right)
    # Run inference with default confidence
    with torch.no_grad():
        pred = model(img)  # Forward pass
        pred_filtered = non_max_suppression(pred, conf_thres=0.25, iou_thres=0.45)
    
    # Get filtered boxes
    boxes_filtered = []
    if len(pred_filtered[0]) > 0:
        for det in pred_filtered[0]:
            x1, y1, x2, y2, conf, cls_id = det
            
            # Convert from padded coordinates to original image coordinates
            if new_width < imgsz or new_height < imgsz:
                # Account for padding
                x1 = min(x1, new_width)
                x2 = min(x2, new_width)
                y1 = min(y1, new_height)
                y2 = min(y2, new_height)
            
            # Convert to original image coordinates
            x1, x2 = x1 / scale, x2 / scale
            y1, y2 = y1 / scale, y2 / scale
            
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            boxes_filtered.append([x1, y1, x2, y2, float(conf), int(cls_id)])
    
    # Draw final boxes
    final_img = original_image.copy()
    
    for box in boxes_filtered:
        x1, y1, x2, y2, conf, cls_id = box
        # Use green color for final detections
        cv2.rectangle(final_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        # Add confidence score
        label = f"{conf:.2f}"
        cv2.putText(final_img, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    axes[1, 1].imshow(cv2.cvtColor(final_img, cv2.COLOR_BGR2RGB))
    axes[1, 1].set_title(f"Final Detections After Filtering: {len(boxes_filtered)}")
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    
    # Save the visualization
    image_filename = Path(image_path).name
    fig_path = os.path.join(output_dir, f"deep_vis_{image_filename.split('.')[0]}.png")
    fig.savefig(fig_path, bbox_inches='tight')
    
    print(f"Total potential boxes: {len(anchor_points) if 'anchor_points' in locals() else 'N/A'}")
    print(f"Actual detected boxes (very low conf): {total_boxes}")
    print(f"Final detections: {len(boxes_filtered)}")
    
    return fig, total_boxes

# Run visualization with extremely low confidence to see more boxes
example_image_path = test_images[0]
fig, box_count = visualize_feature_maps_and_boxes(example_image_path, model, conf_threshold=0.00001)
plt.show()

print(f"Found {box_count} candidate boxes total")
print(f"Visualization saved to {output_dir}")

# Try with one more image
if len(test_images) > 1:
    fig, count = visualize_feature_maps_and_boxes(test_images[1], model, conf_threshold=0.00001)
    print(f"Image 2: Found {count} candidate boxes")
    plt.close(fig)

Total potential boxes: 28620
Actual detected boxes (very low conf): 264
Final detections: 2
Found 264 candidate boxes total
Visualization saved to /home/omtalmo/Olaf_TTK4265/TDT4265-Snow-pole-detection/yolov5_our_files/results
Total potential boxes: 28620
Actual detected boxes (very low conf): 270
Final detections: 1
Image 2: Found 270 candidate boxes


### Aspect ratio analysis for 1 image

In [19]:
import os
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

def analyze_box_aspect_ratios(image_path, model, conf_threshold=0.00001):
    """
    Analyze and visualize aspect ratios of the detected bounding boxes.
    
    Args:
        image_path: Path to the image
        model: YOLOv5 model
        conf_threshold: Threshold for box detection
    """
    # Load image
    image = cv2.imread(image_path)
    original_image = image.copy()
    h, w = image.shape[:2]
    
    # Use global device variable
    global device
    
    # Prepare image for YOLOv5 inference
    imgsz = 640  # Default YOLOv5 input size
    
    # Calculate scaling factors
    scale = min(imgsz / w, imgsz / h)
    new_width = int(w * scale)
    new_height = int(h * scale)
    
    # Resize the image
    resized_image = cv2.resize(image, (new_width, new_height))
    
    # Create a blank canvas with the target size
    input_image = np.zeros((imgsz, imgsz, 3), dtype=np.uint8)
    
    # Place the resized image on the canvas
    input_image[:new_height, :new_width, :] = resized_image
    
    # Convert to tensor and normalize
    img = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    img = torch.from_numpy(img).to(device)
    img = img.float() / 255.0  # Normalize 0-1
    if len(img.shape) == 3:
        img = img.permute(2, 0, 1)  # HWC to CHW (batch, channels, height, width)
    img = img.unsqueeze(0)  # Add batch dimension
    
    print("Getting predictions with very low confidence threshold...")
    
    # Run inference with extremely low confidence
    with torch.no_grad():
        pred = model(img)  # Forward pass
        pred_low_conf = non_max_suppression(pred, conf_thres=conf_threshold, iou_thres=0.45)
    
    # Get all boxes
    boxes_low_conf = []
    if len(pred_low_conf[0]) > 0:
        for det in pred_low_conf[0]:
            x1, y1, x2, y2, conf, cls_id = det
            
            # Convert from padded coordinates to original image coordinates
            if new_width < imgsz or new_height < imgsz:
                # Account for padding
                x1 = min(x1, new_width)
                x2 = min(x2, new_width)
                y1 = min(y1, new_height)
                y2 = min(y2, new_height)
            
            # Convert to original image coordinates
            x1, x2 = x1 / scale, x2 / scale
            y1, y2 = y1 / scale, y2 / scale
            
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            boxes_low_conf.append([x1, y1, x2, y2, float(conf), int(cls_id)])
    
    # Calculate aspect ratios for all boxes
    aspect_ratios = []
    for box in boxes_low_conf:
        x1, y1, x2, y2 = box[:4]
        width = x2 - x1
        height = y2 - y1
        # Avoid division by zero
        if height == 0:
            continue
        aspect_ratio = width / height
        aspect_ratios.append(aspect_ratio)
    
    # Define aspect ratio categories
    categories = {
        'very_tall': (0, 0.33),     # height is 3x+ width
        'tall': (0.33, 0.67),       # height is 1.5x to 3x width
        'slightly_tall': (0.67, 0.9), # height slightly larger than width
        'square': (0.9, 1.1),       # roughly square
        'slightly_wide': (1.1, 1.5), # width slightly larger than height
        'wide': (1.5, 3.0),        # width is 1.5x to 3x height
        'very_wide': (3.0, float('inf')) # width is 3x+ height
    }
    
    # Categorize boxes
    categorized = defaultdict(list)
    for ar in aspect_ratios:
        for category, (min_ar, max_ar) in categories.items():
            if min_ar <= ar < max_ar:
                categorized[category].append(ar)
                break
    
    # Count categories
    category_counts = {category: len(boxes) for category, boxes in categorized.items()}
    total_boxes = len(aspect_ratios)
    
    # Create figure for visualization
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(f"Bounding Box Aspect Ratio Analysis: {Path(image_path).name}", fontsize=16)
    
    # 1. Original image with all bounding boxes colored by aspect ratio
    # Create a copy of the original image
    aspect_ratio_image = original_image.copy()
    
    # Color mapping for categories
    color_map = {
        'very_tall': (0, 0, 255),     # Red
        'tall': (0, 127, 255),         # Orange
        'slightly_tall': (0, 255, 255), # Yellow
        'square': (0, 255, 0),         # Green
        'slightly_wide': (255, 255, 0), # Cyan
        'wide': (255, 0, 0),           # Blue
        'very_wide': (255, 0, 255)      # Magenta
    }
    
    # Draw boxes with category-based coloring
    for box in boxes_low_conf:
        x1, y1, x2, y2 = map(int, box[:4])
        width = x2 - x1
        height = y2 - y1
        
        # Avoid division by zero
        if height == 0:
            continue
            
        aspect_ratio = width / height
        
        # Determine category
        box_category = None
        for category, (min_ar, max_ar) in categories.items():
            if min_ar <= aspect_ratio < max_ar:
                box_category = category
                break
                
        # Get color based on category
        color = color_map.get(box_category, (255, 255, 255))
        
        # Draw rectangle
        cv2.rectangle(aspect_ratio_image, (x1, y1), (x2, y2), color, 2)
    
    # Display image
    axes[0].imshow(cv2.cvtColor(aspect_ratio_image, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Boxes by Aspect Ratio Category (total: {total_boxes})")
    axes[0].axis('off')
    
    # Create legend for categories
    legend_elements = []
    for category, color in color_map.items():
        # Convert BGR to RGB for matplotlib
        color_rgb = (color[2]/255, color[1]/255, color[0]/255)
        count = len(categorized[category])
        percentage = (count / total_boxes * 100) if total_boxes > 0 else 0
        
        # Create patch for legend
        patch = plt.Rectangle((0, 0), 1, 1, facecolor=color_rgb, edgecolor='black')
        legend_elements.append((patch, f"{category}: {count} boxes ({percentage:.1f}%)"))
    
    # Add legend to first subplot
    legend_patches = [item[0] for item in legend_elements]
    legend_labels = [item[1] for item in legend_elements]
    axes[0].legend(legend_patches, legend_labels, loc='upper right')
    
    # 2. Histogram of aspect ratios
    axes[1].hist(aspect_ratios, bins=50, range=(0, 5), alpha=0.7, color='blue')
    axes[1].set_title(f"Distribution of Aspect Ratios (width/height)")
    axes[1].set_xlabel("Aspect Ratio (width/height)")
    axes[1].set_ylabel("Number of Boxes")
    axes[1].grid(alpha=0.3)
    
    # Add vertical lines for category boundaries
    colors = ['r', 'orange', 'y', 'g', 'c', 'b', 'm']
    for i, (category, (min_val, max_val)) in enumerate(categories.items()):
        if min_val > 0:
            axes[1].axvline(x=min_val, color=colors[i % len(colors)], linestyle='--', alpha=0.5)
    
    # Add vertical line at aspect ratio = 1 (square)
    axes[1].axvline(x=1.0, color='g', linestyle='-', alpha=0.8, label="Square (AR=1)")
    
    # Add text showing statistics
    stats_text = "\n".join([
        f"Total boxes: {total_boxes}",
        f"Mean aspect ratio: {np.mean(aspect_ratios):.2f}",
        f"Median aspect ratio: {np.median(aspect_ratios):.2f}",
        f"Min aspect ratio: {min(aspect_ratios):.2f}",
        f"Max aspect ratio: {max(aspect_ratios):.2f}"
    ])
    axes[1].text(0.95, 0.95, stats_text, 
                transform=axes[1].transAxes, 
                fontsize=10, 
                bbox=dict(facecolor='white', alpha=0.7),
                verticalalignment='top', 
                horizontalalignment='right')
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    
    # Save the visualization
    image_filename = Path(image_path).name
    fig_path = os.path.join(output_dir, f"aspect_ratio_analysis_{image_filename.split('.')[0]}.png")
    fig.savefig(fig_path, bbox_inches='tight')
    
    # Print summary statistics
    print("\n=== Bounding Box Aspect Ratio Analysis ===")
    print(f"Total boxes analyzed: {total_boxes}")
    print("\nCategory distribution:")
    for category, count in category_counts.items():
        percentage = (count / total_boxes * 100) if total_boxes > 0 else 0
        min_val, max_val = categories[category]
        print(f"  {category}: {count} boxes ({percentage:.1f}%) - AR range: {min_val:.2f} to {max_val:.2f}")
    
    print("\nStatistical summary:")
    print(f"  Mean aspect ratio: {np.mean(aspect_ratios):.2f}")
    print(f"  Median aspect ratio: {np.median(aspect_ratios):.2f}")
    print(f"  Standard deviation: {np.std(aspect_ratios):.2f}")
    print(f"  Min aspect ratio: {min(aspect_ratios):.2f}")
    print(f"  Max aspect ratio: {max(aspect_ratios):.2f}")
    
    return aspect_ratios, categorized, fig

# Run the aspect ratio analysis
example_image_path = test_images[0]
aspect_ratios, categorized_boxes, fig = analyze_box_aspect_ratios(example_image_path, model, conf_threshold=0.00001)
plt.show()

# Try with one more image if available
if len(test_images) > 1:
    aspect_ratios2, categorized_boxes2, fig2 = analyze_box_aspect_ratios(test_images[1], model, conf_threshold=0.00001)
    plt.show()

Getting predictions with very low confidence threshold...

=== Bounding Box Aspect Ratio Analysis ===
Total boxes analyzed: 102

Category distribution:
  very_tall: 71 boxes (69.6%) - AR range: 0.00 to 0.33
  tall: 25 boxes (24.5%) - AR range: 0.33 to 0.67
  slightly_tall: 5 boxes (4.9%) - AR range: 0.67 to 0.90
  slightly_wide: 1 boxes (1.0%) - AR range: 1.10 to 1.50

Statistical summary:
  Mean aspect ratio: 0.27
  Median aspect ratio: 0.19
  Standard deviation: 0.20
  Min aspect ratio: 0.08
  Max aspect ratio: 1.12
Getting predictions with very low confidence threshold...

=== Bounding Box Aspect Ratio Analysis ===
Total boxes analyzed: 107

Category distribution:
  very_tall: 82 boxes (76.6%) - AR range: 0.00 to 0.33
  tall: 19 boxes (17.8%) - AR range: 0.33 to 0.67
  slightly_tall: 6 boxes (5.6%) - AR range: 0.67 to 0.90

Statistical summary:
  Mean aspect ratio: 0.26
  Median aspect ratio: 0.19
  Standard deviation: 0.19
  Min aspect ratio: 0.07
  Max aspect ratio: 0.90
